In [ ]:
# ═══════════════════════════════════════════════════════════════
# NOTEBOOK: Gemini Dissent Labeling Pipeline
# Runtime: CPU (no GPU needed!) — change via Runtime > Change runtime type
# ═══════════════════════════════════════════════════════════════

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Install & Configure Gemini via Vertex AI — GLOBAL ENDPOINT
# ══════════════════════════════════════════════════════════════════════

!pip -q install -U google-genai nest_asyncio

import os, json, time, datetime, traceback, math, gc
import pandas as pd

from google.colab import auth
auth.authenticate_user()

from google import genai

PROJECT_ID = "" # REPLACE

# ═══════════════════════════════════════════════════════════
# GLOBAL endpoint — Google routes to least-loaded region
# This alone can 2-3x your effective throughput
# ═══════════════════════════════════════════════════════════
client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location="global",           # ← CHANGED from us-central1
)

# Quick verify
test_response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents="Reply with exactly: CONNECTION OK",
    config={
        "temperature": 0,
        "max_output_tokens": 100,
        "thinking_config": {"thinking_budget": 0},
    },
)

response_text = test_response.text if test_response.text else "(empty)"
print(f"✅ Vertex AI connected: {response_text.strip()}")
print(f"   Project: {PROJECT_ID}")
print(f"   Location: GLOBAL (auto-routes to best region)")

LABELS = {
    "substantive_dissent": 0,
    "agreement": 1,
    "neutral": 2,
    "social_disagreement": 3,
}
LABEL_NAMES = {v: k for k, v in LABELS.items()}
print(f"   Label schema: {LABELS}")

✅ Vertex AI connected: CONNECTION OK
   Project: project-41aa31b7-2463-46fd-963
   Location: GLOBAL (auto-routes to best region)
   Label schema: {'substantive_dissent': 0, 'agreement': 1, 'neutral': 2, 'social_disagreement': 3}


In [ ]:
# Check your actual quota in the Cloud Console:
print("Go to: https://console.cloud.google.com/iam-admin/quotas")
print(f"Filter by: 'gemini' and project '{PROJECT_ID}'")
print("Look for 'Generate content requests per minute per region'")

Go to: https://console.cloud.google.com/iam-admin/quotas
Filter by: 'gemini' and project 'project-41aa31b7-2463-46fd-963'
Look for 'Generate content requests per minute per region'


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Helper functions (error logging, atomic saves)
# ══════════════════════════════════════════════════════════════════════

def log_error(output_dir, chunk_name, err):
    os.makedirs(output_dir, exist_ok=True)
    log_path = os.path.join(output_dir, "gemini_chunk_errors.txt")
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"{datetime.datetime.now().isoformat()} | {chunk_name}\n")
        f.write(f"{repr(err)}\n")
        f.write(traceback.format_exc())
        f.write("\n" + "=" * 100 + "\n")
    print("Logged error to:", log_path)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 6: Batch Prediction Setup — FIXED metadata format
# ══════════════════════════════════════════════════════════════════

!pip -q install google-cloud-storage

from google.cloud import storage
import json

SYSTEM_INSTRUCTION = """You are a precise stance classifier for Reddit conversations.
You will receive a parent comment and a reply. Classify the reply into exactly ONE category.

CATEGORIES:
1. SUBSTANTIVE_DISSENT — The reply challenges, contradicts, or argues against the parent's
   core claim with reasoning, evidence, a counter-example, or a counter-perspective.
2. AGREEMENT — The reply supports, endorses, validates, or builds upon the parent's position.
3. NEUTRAL — The reply asks a clarifying question, provides factual information without
   taking a stance, changes the subject, or is tangential to the parent's core argument.
4. SOCIAL_DISAGREEMENT — The reply expresses hostility, mockery, dismissal, or personal
   attack WITHOUT engaging the substance of the parent's argument.

RULES:
- If the reply contains BOTH substance and hostility, choose SUBSTANTIVE_DISSENT.
- If the parent is empty or the reply is clearly unrelated, choose NEUTRAL.
- Respond with ONLY a valid JSON object.

OUTPUT FORMAT (strict JSON, no markdown):
{"label": "SUBSTANTIVE_DISSENT", "confidence": 0.85}"""

BUCKET_NAME = f"{PROJECT_ID}-dissent-batch"
GCS_INPUT_PREFIX = "batch_input_v2"
GCS_OUTPUT_PREFIX = "batch_output_v2"

LABELS = {
    "substantive_dissent": 0,
    "agreement": 1,
    "neutral": 2,
    "social_disagreement": 3,
}


def create_bucket_if_needed(bucket_name):
    storage_client = storage.Client(project=PROJECT_ID)
    try:
        bucket = storage_client.get_bucket(bucket_name)
        print(f"  Bucket exists: gs://{bucket_name}")
    except Exception:
        bucket = storage_client.create_bucket(bucket_name, location="us-central1")
        print(f"  ✅ Created bucket: gs://{bucket_name}")
    return bucket


def smart_truncate(text, max_len=4000):
    """Truncate to the last sentence boundary within max_len chars."""
    text = str(text).strip()
    if len(text) <= max_len:
        return text
    truncated = text[:max_len]
    # Try to cut at a sentence boundary in the back half
    for sep in ['. ', '! ', '? ', '\n']:
        last_idx = truncated.rfind(sep)
        if last_idx > max_len * 0.5:
            return truncated[:last_idx + 1].strip()
    # No good boundary found — cut at last space to avoid mid-word
    last_space = truncated.rfind(' ')
    if last_space > max_len * 0.5:
        return truncated[:last_space].strip() + "..."
    return truncated.strip() + "..."


def prepare_batch_jsonl(chunk_df):
    lines = []
    for row_num, (idx, row) in enumerate(chunk_df.iterrows()):
        parent = smart_truncate(row.get("parent_body", ""))
        reply  = smart_truncate(row.get("comment_body", ""))

        if not reply or reply in ('[deleted]', '[removed]', '', 'nan'):
            continue

        request = {
            "request": {
                "model": "publishers/google/models/gemini-2.5-flash-lite",
                "contents": [
                    {"role": "user", "parts": [{"text": f"Parent comment: {parent}\n\nReply: {reply}"}]}
                ],
                "systemInstruction": {
                    "parts": [{"text": SYSTEM_INSTRUCTION}]
                },
                "generationConfig": {
                    "temperature": 0.1,
                    "maxOutputTokens": 100,
                    "responseMimeType": "application/json",
                },
            },
            "metadata": str(row_num),
        }
        lines.append(json.dumps(request))

    return "\n".join(lines)


def upload_to_gcs(bucket_name, blob_name, content):
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.upload_from_string(content, content_type="application/jsonl")
    uri = f"gs://{bucket_name}/{blob_name}"
    print(f"  Uploaded: {uri} ({len(content)/1e6:.1f} MB, {content.count(chr(10))+1:,} requests)")
    return uri


def submit_batch_job(input_uri, output_uri_prefix):
    response = client.batches.create(
        model="publishers/google/models/gemini-2.5-flash-lite",
        src=input_uri,
        config={"dest": output_uri_prefix},
    )
    print(f"  ✅ Job submitted: {response.name}")
    print(f"     State: {response.state}")
    return response


print("✅ Batch functions defined (v2 — metadata fix).")
print(f"   Bucket: gs://{BUCKET_NAME}")
print(f"   Input prefix: {GCS_INPUT_PREFIX}")
print(f"   Output prefix: {GCS_OUTPUT_PREFIX}")

✅ Batch functions defined (v2 — metadata fix).
   Bucket: gs://project-41aa31b7-2463-46fd-963-dissent-batch
   Input prefix: batch_input_v2
   Output prefix: batch_output_v2


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 7: Submit all chunks as batch jobs — then walk away
# ══════════════════════════════════════════════════════════════════════

ALL_SUBS_CONFIG = {
    "POLOP":     "politicalopinions.csv",
    "T10D":      "the10thdentist.csv",
    "UNPOPULAR": "unpopularopinion.csv",
    "CMV":       "changemyview.csv",
    "AITA":      "amitheasshole.csv",
}

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
MERGED_CSV_DIR = os.path.join(BASE_DIR, "merged_subreddits")
CHUNKS_DIR = os.path.join(BASE_DIR, "chunks")
TARGET_ROWS_PER_CHUNK = 50_000

create_bucket_if_needed(BUCKET_NAME)
batch_jobs = []

for current_sub, csv_filename in ALL_SUBS_CONFIG.items():
    merged_csv_path = os.path.join(MERGED_CSV_DIR, csv_filename)
    sub_chunk_dir = os.path.join(CHUNKS_DIR, f"{current_sub}_chunks")
    os.makedirs(sub_chunk_dir, exist_ok=True)

    print(f"\n{'═' * 60}")
    print(f"  SUBREDDIT: {current_sub}")
    print(f"{'═' * 60}")

    chunk_files = sorted([
        f for f in os.listdir(sub_chunk_dir)
        if f.startswith(f"{current_sub}_chunk_") and f.endswith(".csv")
        and "_labeled" not in f
    ])

    if not chunk_files:
        if not os.path.exists(merged_csv_path):
            print(f"  ⚠️  Not found: {merged_csv_path}")
            continue

        print(f"  Chunking {csv_filename}...")
        df = pd.read_csv(merged_csv_path, low_memory=False)
        print(f"  Total rows: {len(df):,}")

        thread_col = "post_id" if "post_id" in df.columns else "link_id"
        thread_ids = df[thread_col].unique()
        thread_to_size = df.groupby(thread_col, sort=False).size()

        chunks_plan, cur, cur_size = [], [], 0
        for tid in thread_ids:
            t_size = thread_to_size[tid]
            if cur_size + t_size > TARGET_ROWS_PER_CHUNK and cur_size > 0:
                chunks_plan.append(cur)
                cur, cur_size = [], 0
            cur.append(tid)
            cur_size += t_size
        if cur:
            chunks_plan.append(cur)

        for i, tids in enumerate(chunks_plan):
            chunk_df = df[df[thread_col].isin(set(tids))]
            out = os.path.join(sub_chunk_dir, f"{current_sub}_chunk_{i:03d}.csv")
            chunk_df.to_csv(out, index=False)
            print(f"    Chunk {i:03d}: {len(chunk_df):,} rows")
        del df; gc.collect()

        chunk_files = sorted([
            f for f in os.listdir(sub_chunk_dir)
            if f.startswith(f"{current_sub}_chunk_") and f.endswith(".csv")
            and "_labeled" not in f
        ])

    for chunk_filename in chunk_files:
        chunk_path = os.path.join(sub_chunk_dir, chunk_filename)
        chunk_name = chunk_filename.replace(".csv", "")

        print(f"\n  📦 {chunk_filename}")
        chunk_df = pd.read_csv(chunk_path, low_memory=False)

        jsonl_content = prepare_batch_jsonl(chunk_df)
        input_blob = f"{GCS_INPUT_PREFIX}/{chunk_name}.jsonl"
        input_uri = upload_to_gcs(BUCKET_NAME, input_blob, jsonl_content)

        output_uri = f"gs://{BUCKET_NAME}/{GCS_OUTPUT_PREFIX}/{chunk_name}/"
        try:
            job = submit_batch_job(input_uri, output_uri)
            batch_jobs.append({
                "sub": current_sub,
                "chunk": chunk_name,
                "job_name": job.name,
                "state": str(job.state),
            })
        except Exception as e:
            print(f"  ❌ Failed to submit: {e}")

        del chunk_df; gc.collect()

print(f"\n{'═' * 60}")
print(f"  ALL JOBS SUBMITTED: {len(batch_jobs)} total")
print(f"{'═' * 60}")
print(f"\n🔗 Monitor: https://console.cloud.google.com/vertex-ai/batch-predictions?project={PROJECT_ID}")
print(f"\n🚿 You can close this notebook and go shower.")
print(f"   Jobs run on Google's servers. Come back in a few hours.")

jobs_path = os.path.join(BASE_DIR, "batch_jobs.json")
with open(jobs_path, "w") as f:
    json.dump(batch_jobs, f, indent=2)
print(f"\n   Job list saved: {jobs_path}")

  Bucket exists: gs://project-41aa31b7-2463-46fd-963-dissent-batch

════════════════════════════════════════════════════════════
  SUBREDDIT: POLOP
════════════════════════════════════════════════════════════
  Chunking politicalopinions.csv...
  Total rows: 86,493
    Chunk 000: 49,998 rows
    Chunk 001: 36,495 rows

  📦 POLOP_chunk_000.csv
  Uploaded: gs://project-41aa31b7-2463-46fd-963-dissent-batch/batch_input_v2/POLOP_chunk_000.jsonl (142.2 MB, 49,998 requests)
  ✅ Job submitted: projects/92175944845/locations/global/batchPredictionJobs/6170059582602215424
     State: JobState.JOB_STATE_PENDING

  📦 POLOP_chunk_001.csv
  Uploaded: gs://project-41aa31b7-2463-46fd-963-dissent-batch/batch_input_v2/POLOP_chunk_001.jsonl (104.1 MB, 36,495 requests)
  ✅ Job submitted: projects/92175944845/locations/global/batchPredictionJobs/981912811871404032
     State: JobState.JOB_STATE_PENDING

════════════════════════════════════════════════════════════
  SUBREDDIT: T10D
═════════════════════════

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DANGER ZONE: Uncomment to delete labeled output files and re-harvest
# ══════════════════════════════════════════════════════════════════════
'''
import os
BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
labeled_dir = os.path.join(BASE_DIR, "labeled_chunks")
count = 0
for root, dirs, files in os.walk(labeled_dir):
    for f in files:
        os.remove(os.path.join(root, f))
        count += 1
        print(f"  🗑️  {f}")
print(f"✅ Deleted {count} files. Now run the harvest cell.")
'''

'\nimport os\nBASE_DIR = "/content/drive/MyDrive/My_Dissent_project"\nlabeled_dir = os.path.join(BASE_DIR, "labeled_chunks")\ncount = 0\nfor root, dirs, files in os.walk(labeled_dir):\n    for f in files:\n        os.remove(os.path.join(root, f))\n        count += 1\n        print(f"  🗑️  {f}")\nprint(f"✅ Deleted {count} files. Now run the harvest cell.")\n'

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CHECK STATUS — Run this whenever you want to see progress
# ══════════════════════════════════════════════════════════════════════

import json

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
jobs_path = os.path.join(BASE_DIR, "batch_jobs.json")
with open(jobs_path) as f:
    batch_jobs = json.load(f)

n_total_jobs = len(batch_jobs)

from collections import Counter
status_counts = Counter()

for job_info in batch_jobs:
    job = client.batches.get(name=job_info["job_name"])
    state = str(job.state).split(".")[-1]
    status_counts[state] += 1

    if state != "JOB_STATE_SUCCEEDED":
        print(f"  {job_info['chunk']:30s} → {state}")

print(f"\n{'═' * 40}")
print(f"  ✅ Succeeded: {status_counts.get('JOB_STATE_SUCCEEDED', 0)}/{n_total_jobs}")
print(f"  ⏳ Pending:   {status_counts.get('JOB_STATE_PENDING', 0)}/{n_total_jobs}")
print(f"  🔄 Running:   {status_counts.get('JOB_STATE_RUNNING', 0)}/{n_total_jobs}")
print(f"  ❌ Failed:    {status_counts.get('JOB_STATE_FAILED', 0)}/{n_total_jobs}")

if status_counts.get("JOB_STATE_SUCCEEDED", 0) == n_total_jobs:
    print(f"\n  🎉 ALL DONE! Run the harvest cell next.")

  POLOP_chunk_000                → JOB_STATE_FAILED
  POLOP_chunk_001                → JOB_STATE_FAILED
  T10D_chunk_000                 → JOB_STATE_FAILED
  T10D_chunk_001                 → JOB_STATE_FAILED
  T10D_chunk_002                 → JOB_STATE_FAILED
  T10D_chunk_003                 → JOB_STATE_FAILED
  T10D_chunk_004                 → JOB_STATE_FAILED
  T10D_chunk_005                 → JOB_STATE_FAILED
  T10D_chunk_006                 → JOB_STATE_FAILED
  T10D_chunk_007                 → JOB_STATE_FAILED
  T10D_chunk_008                 → JOB_STATE_FAILED
  T10D_chunk_009                 → JOB_STATE_FAILED
  T10D_chunk_010                 → JOB_STATE_FAILED
  T10D_chunk_011                 → JOB_STATE_FAILED
  UNPOPULAR_chunk_000            → JOB_STATE_FAILED
  UNPOPULAR_chunk_001            → JOB_STATE_FAILED
  UNPOPULAR_chunk_002            → JOB_STATE_FAILED
  UNPOPULAR_chunk_003            → JOB_STATE_FAILED
  UNPOPULAR_chunk_004            → JOB_STATE_FAILED
  UNPOPULAR_

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DETAILED PROGRESS — see how many rows each job has completed
# ══════════════════════════════════════════════════════════════════

import json

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
jobs_path = os.path.join(BASE_DIR, "batch_jobs.json")
with open(jobs_path) as f:
    batch_jobs = json.load(f)

total_succeeded = 0
total_incomplete = 0
total_failed_rows = 0
jobs_done = 0

for job_info in batch_jobs:
    job = client.batches.get(name=job_info["job_name"])
    state = str(job.state).split(".")[-1]

    stats = job.completion_stats
    s = stats.successful_count or 0 if stats else 0
    i = stats.incomplete_count or 0 if stats else 0
    f_count = stats.failed_count or 0 if stats else 0

    total_succeeded += s
    total_incomplete += i
    total_failed_rows += f_count

    if state == "JOB_STATE_SUCCEEDED":
        jobs_done += 1
    else:
        pct = s / (s + i + f_count) * 100 if (s + i + f_count) > 0 else 0
        print(f"  {job_info['chunk']:30s} {state:30s} {s:>7,}/{s+i+f_count:>7,} ({pct:5.1f}%)")

print(f"\n{'═' * 60}")
print(f"  Jobs completed:    {jobs_done}/50")
print(f"  Rows succeeded:    {total_succeeded:,}")
print(f"  Rows incomplete:   {total_incomplete:,}")
print(f"  Rows failed:       {total_failed_rows:,}")
print(f"  Total progress:    {total_succeeded/(total_succeeded+total_incomplete+total_failed_rows)*100:.1f}%")
print(f"{'═' * 60}")

  POLOP_chunk_000                JOB_STATE_FAILED                41,576/ 49,998 ( 83.2%)
  POLOP_chunk_001                JOB_STATE_FAILED                29,722/ 36,495 ( 81.4%)
  T10D_chunk_000                 JOB_STATE_FAILED                39,200/ 49,864 ( 78.6%)
  T10D_chunk_001                 JOB_STATE_FAILED                39,451/ 49,933 ( 79.0%)
  T10D_chunk_002                 JOB_STATE_FAILED                38,973/ 49,955 ( 78.0%)
  T10D_chunk_003                 JOB_STATE_FAILED                39,413/ 49,870 ( 79.0%)
  T10D_chunk_004                 JOB_STATE_FAILED                39,187/ 49,998 ( 78.4%)
  T10D_chunk_005                 JOB_STATE_FAILED                39,061/ 49,770 ( 78.5%)
  T10D_chunk_006                 JOB_STATE_FAILED                39,418/ 49,902 ( 79.0%)
  T10D_chunk_007                 JOB_STATE_FAILED                39,099/ 49,996 ( 78.2%)
  T10D_chunk_008                 JOB_STATE_FAILED                39,346/ 49,911 ( 78.8%)
  T10D_chunk_009     

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# GET THE ACTUAL ERROR — don't guess, just read it
# ══════════════════════════════════════════════════════════════════════

import json

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
jobs_path = os.path.join(BASE_DIR, "batch_jobs.json")
with open(jobs_path) as f:
    batch_jobs = json.load(f)

# Just check the first failed job
job = client.batches.get(name=batch_jobs[0]["job_name"])

print(f"Job: {batch_jobs[0]['chunk']}")
print(f"State: {job.state}")
print(f"\nFull job object attributes:")
for attr in dir(job):
    if not attr.startswith("_"):
        try:
            val = getattr(job, attr)
            if val and not callable(val):
                print(f"  {attr}: {val}")
        except:
            pass

Job: POLOP_chunk_000
State: JobState.JOB_STATE_FAILED

Full job object attributes:
  completion_stats: failed_count=None incomplete_count=8422 successful_count=41576 successful_forecast_point_count=None
  create_time: 2026-04-19 19:17:53.667793+00:00
  dest: format='jsonl' gcs_uri='gs://project-41aa31b7-2463-46fd-963-dissent-batch/batch_output_v2/POLOP_chunk_000/' bigquery_uri=None file_name=None inlined_responses=None inlined_embed_content_responses=None
  display_name: genai_batch_job_20260419191752_d2cc1
  done: True
  end_time: 2026-04-20 19:19:34.368625+00:00
  error: details=None code=4 message='Deadline exceeded due to job running for maximum allowed duration of 24 hours. Please retry the unprocessed rows or start over with a smaller batch size.'
  model: publishers/google/models/gemini-2.5-flash-lite
  model_config: {'alias_generator': <function to_camel at 0x78cd6bb849a0>, 'populate_by_name': True, 'from_attributes': True, 'protected_namespaces': (), 'extra': 'forbid', 'arbitr

/tmp/ipykernel_19703/2127600443.py:21: PydanticDeprecatedSince211: Accessing the 'model_computed_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  val = getattr(job, attr)
/tmp/ipykernel_19703/2127600443.py:21: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  val = getattr(job, attr)


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# HARVEST: Download batch results from GCS, parse, and save labeled CSVs
# ══════════════════════════════════════════════════════════════════════
# Run this AFTER all batch jobs show JOB_STATE_SUCCEEDED.
# Safe to re-run: skips chunks that already have a _labeled.csv file.

import json
import re
from google.cloud import storage

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
CHUNKS_DIR = os.path.join(BASE_DIR, "chunks")
LABELED_CHUNKS_DIR = os.path.join(BASE_DIR, "labeled_chunks")

jobs_path = os.path.join(BASE_DIR, "batch_jobs.json")
with open(jobs_path) as f:
    batch_jobs = json.load(f)

LABEL_MAP = {
    "substantive_dissent": 0,
    "agreement": 1,
    "neutral": 2,
    "social_disagreement": 3,
}
LABEL_NAME_MAP = {v: k for k, v in LABEL_MAP.items()}

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

success_count = 0
skip_count = 0
fail_count = 0

for job_info in batch_jobs:
    sub = job_info["sub"]
    chunk_name = job_info["chunk"]
    job_name = job_info["job_name"]

    # ── Output paths ──
    labeled_sub_dir = os.path.join(LABELED_CHUNKS_DIR, f"{sub}_labeled")
    os.makedirs(labeled_sub_dir, exist_ok=True)
    labeled_path = os.path.join(labeled_sub_dir, f"{chunk_name}_labeled.csv")

    # ── Skip if already harvested ──
    if os.path.exists(labeled_path) and os.path.getsize(labeled_path) > 100:
        skip_count += 1
        continue

    # ── Check job status ──
    job = client.batches.get(name=job_name)
    state = str(job.state).split(".")[-1]
    if state not in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED"):
        print(f"  ⏭️  {chunk_name}: {state} — still running, skipping")
        fail_count += 1
        continue

    # ── Load the original chunk CSV ──
    chunk_csv_path = os.path.join(CHUNKS_DIR, f"{sub}_chunks", f"{chunk_name}.csv")
    if not os.path.exists(chunk_csv_path):
        print(f"  ❌ {chunk_name}: original chunk CSV not found at {chunk_csv_path}")
        fail_count += 1
        continue

    chunk_df = pd.read_csv(chunk_csv_path, low_memory=False)

    # ── Download JSONL results from GCS ──
    output_prefix = f"{GCS_OUTPUT_PREFIX}/{chunk_name}/"
    blobs = list(bucket.list_blobs(prefix=output_prefix))
    jsonl_blobs = [b for b in blobs if b.name.endswith(".jsonl")]

    if not jsonl_blobs:
        print(f"  ❌ {chunk_name}: no JSONL output found at gs://{BUCKET_NAME}/{output_prefix}")
        fail_count += 1
        continue

    # Parse all response lines
    labels = {}  # row_index -> {"label": ..., "confidence": ...}
    parse_errors = 0

    for blob in jsonl_blobs:
        content = blob.download_as_text()
        for line in content.strip().split("\n"):
            if not line.strip():
                continue
            try:
                obj = json.loads(line)

                # Extract row index from metadata
                row_idx = int(obj.get("metadata", -1))
                if row_idx < 0:
                    parse_errors += 1
                    continue

                # Extract the model's response text
                response = obj.get("response", {})
                candidates = response.get("candidates", [])
                if not candidates:
                    parse_errors += 1
                    continue

                parts = candidates[0].get("content", {}).get("parts", [])
                if not parts:
                    parse_errors += 1
                    continue

                text = parts[0].get("text", "").strip()

                # Parse the JSON response from the model
                parsed = json.loads(text)
                raw_label = parsed.get("label", "neutral").lower().strip()
                confidence = float(parsed.get("confidence", 0.5))

                # Normalize label
                if raw_label not in LABEL_MAP:
                    # Try fuzzy match
                    matched = False
                    for key in LABEL_MAP:
                        if key in raw_label or raw_label in key:
                            raw_label = key
                            matched = True
                            break
                    if not matched:
                        raw_label = "neutral"
                        confidence = 0.0

                labels[row_idx] = {
                    "label": LABEL_MAP[raw_label],
                    "label_name": raw_label,
                    "confidence": confidence,
                }

            except (json.JSONDecodeError, KeyError, ValueError, IndexError):
                parse_errors += 1
                continue

    # ── Merge labels onto chunk ──
    chunk_df["label"] = chunk_df.index.map(lambda i: labels.get(i, {}).get("label", 2))
    chunk_df["label_name"] = chunk_df.index.map(lambda i: labels.get(i, {}).get("label_name", "neutral"))
    chunk_df["confidence"] = chunk_df.index.map(lambda i: labels.get(i, {}).get("confidence", 0.0))

    n_labeled = len(labels)
    n_total = len(chunk_df)
    n_missing = n_total - n_labeled

    # ── Save ──
    chunk_df.to_csv(labeled_path, index=False)
    success_count += 1

    print(f"  ✅ {chunk_name}: {n_labeled:,}/{n_total:,} labeled, "
          f"{n_missing:,} defaulted to neutral, {parse_errors} parse errors → {labeled_path}")

    del chunk_df
    gc.collect()

print(f"\n{'═' * 60}")
print(f"  HARVEST COMPLETE")
print(f"{'═' * 60}")
print(f"  ✅ Saved:   {success_count}")
print(f"  ⏭️  Skipped: {skip_count} (already existed)")
print(f"  ❌ Failed:  {fail_count}")
print(f"\n  Output: {LABELED_CHUNKS_DIR}")

  ✅ AITA_chunk_010: 49,553/49,553 labeled, 0 defaulted to neutral, 0 parse errors → /content/drive/MyDrive/My_Dissent_project/labeled_chunks/AITA_labeled/AITA_chunk_010_labeled.csv
  ✅ AITA_chunk_011: 44,859/44,860 labeled, 1 defaulted to neutral, 1 parse errors → /content/drive/MyDrive/My_Dissent_project/labeled_chunks/AITA_labeled/AITA_chunk_011_labeled.csv

════════════════════════════════════════════════════════════
  HARVEST COMPLETE
════════════════════════════════════════════════════════════
  ✅ Saved:   2
  ⏭️  Skipped: 48 (already existed)
  ❌ Failed:  0

  Output: /content/drive/MyDrive/My_Dissent_project/labeled_chunks


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RETRY: Resubmit only unlabeled rows (confidence == 0.0)
# Submit fewer jobs this time — 10 max concurrent
# ══════════════════════════════════════════════════════════════════════

import json, os, gc
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
LABELED_CHUNKS_DIR = os.path.join(BASE_DIR, "labeled_chunks")
CHUNKS_DIR = os.path.join(BASE_DIR, "chunks")

retry_jobs = []

for sub_dir in sorted(os.listdir(LABELED_CHUNKS_DIR)):
    sub_path = os.path.join(LABELED_CHUNKS_DIR, sub_dir)
    if not os.path.isdir(sub_path):
        continue

    sub_key = sub_dir.replace("_labeled", "")

    for labeled_file in sorted(os.listdir(sub_path)):
        if not labeled_file.endswith("_labeled.csv"):
            continue

        labeled_path = os.path.join(sub_path, labeled_file)
        df = pd.read_csv(labeled_path, low_memory=False)

        # Find rows that weren't labeled (confidence == 0.0 means default/missing)
        missing = df[df["confidence"] == 0.0].copy()

        if len(missing) < 10:
            print(f"  ✅ {labeled_file}: only {len(missing)} missing — skipping retry")
            continue

        print(f"  🔄 {labeled_file}: {len(missing):,} missing out of {len(df):,}")

        # Create retry JSONL
        chunk_name = labeled_file.replace("_labeled.csv", "")
        retry_name = f"{chunk_name}_retry"

        jsonl_content = prepare_batch_jsonl(missing.reset_index(drop=True))

        if not jsonl_content.strip():
            print(f"     No valid rows to retry — skipping")
            continue

        input_blob = f"batch_input_v2/{retry_name}.jsonl"
        input_uri = upload_to_gcs(BUCKET_NAME, input_blob, jsonl_content)

        output_uri = f"gs://{BUCKET_NAME}/{GCS_OUTPUT_PREFIX}/{retry_name}/"

        try:
            job = submit_batch_job(input_uri, output_uri)
            retry_jobs.append({
                "sub": sub_key,
                "chunk": retry_name,
                "original_chunk": chunk_name,
                "job_name": job.name,
                "state": str(job.state),
                "missing_indices": missing.index.tolist(),
            })
        except Exception as e:
            print(f"     ❌ Failed to submit: {e}")

        del df, missing
        gc.collect()

# Save retry job list
retry_path = os.path.join(BASE_DIR, "batch_jobs_retry.json")
with open(retry_path, "w") as f:
    json.dump(retry_jobs, f, indent=2)

print(f"\n{'═' * 60}")
print(f"  RETRY JOBS SUBMITTED: {len(retry_jobs)}")
print(f"  Saved to: {retry_path}")
print(f"{'═' * 60}")
print(f"\n  These are much smaller (~10K rows each) so they'll")
print(f"  finish well within the 24h limit.")

  🔄 AITA_chunk_000_labeled.csv: 11,709 missing out of 48,330
  Uploaded: gs://project-41aa31b7-2463-46fd-963-dissent-batch/batch_input_v2/AITA_chunk_000_retry.jsonl (37.2 MB, 11,709 requests)
  ✅ Job submitted: projects/92175944845/locations/global/batchPredictionJobs/7921493644719161344
     State: JobState.JOB_STATE_PENDING
  🔄 AITA_chunk_001_labeled.csv: 11,234 missing out of 46,592
  Uploaded: gs://project-41aa31b7-2463-46fd-963-dissent-batch/batch_input_v2/AITA_chunk_001_retry.jsonl (32.5 MB, 11,234 requests)
  ✅ Job submitted: projects/92175944845/locations/global/batchPredictionJobs/5863348615010844672
     State: JobState.JOB_STATE_PENDING
  🔄 AITA_chunk_002_labeled.csv: 11,788 missing out of 49,628
  Uploaded: gs://project-41aa31b7-2463-46fd-963-dissent-batch/batch_input_v2/AITA_chunk_002_retry.jsonl (33.6 MB, 11,788 requests)
  ✅ Job submitted: projects/92175944845/locations/global/batchPredictionJobs/6642471350545940480
     State: JobState.JOB_STATE_PENDING
  🔄 AITA_chunk_0

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CHECK STATUS — Both original jobs and retry jobs
# ══════════════════════════════════════════════════════════════════════

import json, os
from collections import Counter

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"

# ── Load both job lists ──
jobs_path = os.path.join(BASE_DIR, "batch_jobs.json")
retry_path = os.path.join(BASE_DIR, "batch_jobs_retry.json")

with open(jobs_path) as f:
    original_jobs = json.load(f)

retry_jobs = []
if os.path.exists(retry_path):
    with open(retry_path) as f:
        retry_jobs = json.load(f)

# ── Original jobs ──
print("═" * 65)
print("  ORIGINAL JOBS (50)")
print("═" * 65)

orig_counts = Counter()
for job_info in original_jobs:
    job = client.batches.get(name=job_info["job_name"])
    state = str(job.state).split(".")[-1]
    orig_counts[state] += 1

    stats = job.completion_stats
    s = stats.successful_count or 0 if stats else 0
    i = stats.incomplete_count or 0 if stats else 0
    f_c = stats.failed_count or 0 if stats else 0
    total = s + i + f_c

    if state != "JOB_STATE_SUCCEEDED":
        pct = s / total * 100 if total > 0 else 0
        print(f"  {job_info['chunk']:30s} {state:30s} {s:>7,}/{total:>7,} ({pct:5.1f}%)")

print(f"\n  ✅ Succeeded: {orig_counts.get('JOB_STATE_SUCCEEDED', 0)}/50")
print(f"  🔄 Running:   {orig_counts.get('JOB_STATE_RUNNING', 0)}/50")
print(f"  ❌ Failed:    {orig_counts.get('JOB_STATE_FAILED', 0)}/50")

# ── Retry jobs ──
if not retry_jobs:
    print(f"\n  ⚠️  No retry jobs found at {retry_path}")
else:
    print(f"\n{'═' * 65}")
    print(f"  RETRY JOBS ({len(retry_jobs)})")
    print("═" * 65)

    retry_counts = Counter()
    retry_rows_done = 0
    retry_rows_total = 0

    for rj in retry_jobs:
        job = client.batches.get(name=rj["job_name"])
        state = str(job.state).split(".")[-1]
        retry_counts[state] += 1

        stats = job.completion_stats
        s = stats.successful_count or 0 if stats else 0
        i = stats.incomplete_count or 0 if stats else 0
        f_c = stats.failed_count or 0 if stats else 0
        total = s + i + f_c
        retry_rows_done += s
        retry_rows_total += total

        if state != "JOB_STATE_SUCCEEDED":
            pct = s / total * 100 if total > 0 else 0
            print(f"  {rj['chunk']:30s} {state:30s} {s:>7,}/{total:>7,} ({pct:5.1f}%)")

    print(f"\n  ✅ Succeeded: {retry_counts.get('JOB_STATE_SUCCEEDED', 0)}/{len(retry_jobs)}")
    print(f"  ⏳ Pending:   {retry_counts.get('JOB_STATE_PENDING', 0)}/{len(retry_jobs)}")
    print(f"  🔄 Running:   {retry_counts.get('JOB_STATE_RUNNING', 0)}/{len(retry_jobs)}")
    print(f"  ❌ Failed:    {retry_counts.get('JOB_STATE_FAILED', 0)}/{len(retry_jobs)}")
    print(f"\n  Retry rows:   {retry_rows_done:,}/{retry_rows_total:,} completed")

    if retry_counts.get("JOB_STATE_SUCCEEDED", 0) == len(retry_jobs):
        print(f"\n  🎉 ALL RETRY JOBS DONE! Run the merge cell next.")

═════════════════════════════════════════════════════════════════
  ORIGINAL JOBS (50)
═════════════════════════════════════════════════════════════════
  POLOP_chunk_000                JOB_STATE_FAILED                41,576/ 49,998 ( 83.2%)
  POLOP_chunk_001                JOB_STATE_FAILED                29,722/ 36,495 ( 81.4%)
  T10D_chunk_000                 JOB_STATE_FAILED                39,200/ 49,864 ( 78.6%)
  T10D_chunk_001                 JOB_STATE_FAILED                39,451/ 49,933 ( 79.0%)
  T10D_chunk_002                 JOB_STATE_FAILED                38,973/ 49,955 ( 78.0%)
  T10D_chunk_003                 JOB_STATE_FAILED                39,413/ 49,870 ( 79.0%)
  T10D_chunk_004                 JOB_STATE_FAILED                39,187/ 49,998 ( 78.4%)
  T10D_chunk_005                 JOB_STATE_FAILED                39,061/ 49,770 ( 78.5%)
  T10D_chunk_006                 JOB_STATE_FAILED                39,418/ 49,902 ( 79.0%)
  T10D_chunk_007                 JOB_STATE_FAI

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MERGE RETRY RESULTS back into the labeled CSVs
# ══════════════════════════════════════════════════════════════════════

import json, os, gc
import pandas as pd
from google.cloud import storage

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
LABELED_CHUNKS_DIR = os.path.join(BASE_DIR, "labeled_chunks")

retry_path = os.path.join(BASE_DIR, "batch_jobs_retry.json")
with open(retry_path) as f:
    retry_jobs = json.load(f)

LABEL_MAP = {
    "substantive_dissent": 0,
    "agreement": 1,
    "neutral": 2,
    "social_disagreement": 3,
}

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

patched = 0
still_missing_total = 0

for rj in retry_jobs:
    sub = rj["sub"]
    retry_name = rj["chunk"]
    original_chunk = rj["original_chunk"]
    job_name = rj["job_name"]
    missing_indices = rj["missing_indices"]

    # Check job status
    job = client.batches.get(name=job_name)
    state = str(job.state).split(".")[-1]
    if state not in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED"):
        print(f"  ⏭️  {retry_name}: {state} — not done yet")
        continue

    # Load the labeled CSV
    labeled_path = os.path.join(
        LABELED_CHUNKS_DIR, f"{sub}_labeled", f"{original_chunk}_labeled.csv"
    )
    if not os.path.exists(labeled_path):
        print(f"  ❌ {original_chunk}: labeled CSV not found")
        continue

    df = pd.read_csv(labeled_path, low_memory=False)

    # Download retry results from GCS
    output_prefix = f"{GCS_OUTPUT_PREFIX}/{retry_name}/"
    blobs = list(bucket.list_blobs(prefix=output_prefix))
    jsonl_blobs = [b for b in blobs if b.name.endswith(".jsonl")]

    if not jsonl_blobs:
        print(f"  ❌ {retry_name}: no output JSONL found")
        continue

    # Parse retry results — row_num in retry maps to missing_indices position
    retry_labels = {}
    parse_errors = 0

    for blob in jsonl_blobs:
        content = blob.download_as_text()
        for line in content.strip().split("\n"):
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
                retry_row_num = int(obj.get("metadata", -1))
                if retry_row_num < 0 or retry_row_num >= len(missing_indices):
                    parse_errors += 1
                    continue

                response = obj.get("response", {})
                candidates = response.get("candidates", [])
                if not candidates:
                    parse_errors += 1
                    continue

                parts = candidates[0].get("content", {}).get("parts", [])
                if not parts:
                    parse_errors += 1
                    continue

                text = parts[0].get("text", "").strip()
                parsed = json.loads(text)
                raw_label = parsed.get("label", "neutral").lower().strip()
                confidence = float(parsed.get("confidence", 0.5))

                if raw_label not in LABEL_MAP:
                    matched = False
                    for key in LABEL_MAP:
                        if key in raw_label or raw_label in key:
                            raw_label = key
                            matched = True
                            break
                    if not matched:
                        raw_label = "neutral"
                        confidence = 0.0

                # Map retry row_num back to original DataFrame index
                original_idx = missing_indices[retry_row_num]
                retry_labels[original_idx] = {
                    "label": LABEL_MAP[raw_label],
                    "label_name": raw_label,
                    "confidence": confidence,
                }
            except (json.JSONDecodeError, KeyError, ValueError, IndexError):
                parse_errors += 1

    # Patch the DataFrame
    filled = 0
    for orig_idx, label_info in retry_labels.items():
        if orig_idx < len(df):
            df.at[orig_idx, "label"] = label_info["label"]
            df.at[orig_idx, "label_name"] = label_info["label_name"]
            df.at[orig_idx, "confidence"] = label_info["confidence"]
            filled += 1

    still_missing = (df["confidence"] == 0.0).sum()
    still_missing_total += still_missing

    # Save back
    df.to_csv(labeled_path, index=False)
    patched += 1

    print(f"  ✅ {original_chunk}: patched {filled:,} rows, "
          f"{still_missing:,} still missing, {parse_errors} parse errors")

    del df
    gc.collect()

print(f"\n{'═' * 60}")
print(f"  MERGE COMPLETE: {patched} chunks patched")
print(f"  Still missing across all chunks: {still_missing_total:,}")
print(f"{'═' * 60}")

  ✅ AITA_chunk_000: patched 11,709 rows, 403 still missing, 0 parse errors
  ✅ AITA_chunk_001: patched 11,234 rows, 531 still missing, 0 parse errors
  ✅ AITA_chunk_002: patched 11,788 rows, 639 still missing, 0 parse errors
  ✅ AITA_chunk_003: patched 11,372 rows, 395 still missing, 0 parse errors
  ✅ AITA_chunk_004: patched 11,819 rows, 361 still missing, 0 parse errors
  ✅ AITA_chunk_005: patched 11,204 rows, 389 still missing, 0 parse errors
  ✅ AITA_chunk_006: patched 12,051 rows, 473 still missing, 0 parse errors
  ✅ AITA_chunk_007: patched 11,812 rows, 317 still missing, 0 parse errors
  ✅ AITA_chunk_008: patched 11,590 rows, 282 still missing, 0 parse errors
  ✅ AITA_chunk_009: patched 11,049 rows, 311 still missing, 0 parse errors
  ✅ CMV_chunk_000: patched 11,659 rows, 361 still missing, 0 parse errors
  ✅ CMV_chunk_001: patched 11,654 rows, 517 still missing, 0 parse errors
  ✅ CMV_chunk_002: patched 11,422 rows, 372 still missing, 0 parse errors
  ✅ CMV_chunk_003: patched 1

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DIAGNOSE & FIX: What's actually still missing?
# ══════════════════════════════════════════════════════════════════════

import json, os, gc
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
LABELED_CHUNKS_DIR = os.path.join(BASE_DIR, "labeled_chunks")

# ── Part 1: Check what "still missing" actually is ──
print("═" * 65)
print("  DIAGNOSIS: What are the confidence==0.0 rows?")
print("═" * 65)

total_rows = 0
total_missing = 0
total_deleted = 0
total_truly_missing = 0

for sub_dir in sorted(os.listdir(LABELED_CHUNKS_DIR)):
    sub_path = os.path.join(LABELED_CHUNKS_DIR, sub_dir)
    if not os.path.isdir(sub_path):
        continue

    for f in sorted(os.listdir(sub_path)):
        if not f.endswith("_labeled.csv"):
            continue

        df = pd.read_csv(os.path.join(sub_path, f), low_memory=False)
        total_rows += len(df)

        missing = df[df["confidence"] == 0.0]
        n_missing = len(missing)
        total_missing += n_missing

        if n_missing == 0:
            continue

        # Check how many are deleted/removed/empty
        deleted_mask = missing["comment_body"].apply(
            lambda x: str(x).strip() in ('[deleted]', '[removed]', '', 'nan', 'NaN')
            or pd.isna(x)
        )
        n_deleted = deleted_mask.sum()
        n_truly_missing = n_missing - n_deleted
        total_deleted += n_deleted
        total_truly_missing += n_truly_missing

        if n_truly_missing > 0:
            print(f"  ⚠️  {f}: {n_missing:,} missing "
                  f"({n_deleted} deleted/empty, {n_truly_missing} REAL gaps)")
        del df
        gc.collect()

print(f"\n{'─' * 65}")
print(f"  Total rows across all labeled chunks: {total_rows:,}")
print(f"  Total confidence==0.0:                {total_missing:,}")
print(f"    └─ Deleted/removed/empty comments:  {total_deleted:,} (expected)")
print(f"    └─ Real unlabeled gaps:             {total_truly_missing:,}")
print(f"  Coverage: {(total_rows - total_truly_missing) / total_rows * 100:.2f}%")
print(f"{'─' * 65}")

# ── Part 2: Check if AITA_chunk_010 and _011 still need harvesting ──
print(f"\n{'═' * 65}")
print("  CHECKING UNHARVESTED CHUNKS (AITA_chunk_010, _011)")
print("═" * 65)

aita_labeled_dir = os.path.join(LABELED_CHUNKS_DIR, "AITA_labeled")
for chunk_name in ["AITA_chunk_010", "AITA_chunk_011"]:
    labeled_path = os.path.join(aita_labeled_dir, f"{chunk_name}_labeled.csv")
    if os.path.exists(labeled_path):
        df = pd.read_csv(labeled_path, low_memory=False)
        n_zero = (df["confidence"] == 0.0).sum()
        print(f"  {chunk_name}: EXISTS ({len(df):,} rows, {n_zero:,} conf==0.0)")
        del df
    else:
        print(f"  {chunk_name}: ❌ NOT FOUND — needs harvesting!")

═════════════════════════════════════════════════════════════════
  DIAGNOSIS: What are the confidence==0.0 rows?
═════════════════════════════════════════════════════════════════
  ⚠️  AITA_chunk_000_labeled.csv: 403 missing (0 deleted/empty, 403 REAL gaps)
  ⚠️  AITA_chunk_001_labeled.csv: 531 missing (0 deleted/empty, 531 REAL gaps)
  ⚠️  AITA_chunk_002_labeled.csv: 639 missing (0 deleted/empty, 639 REAL gaps)
  ⚠️  AITA_chunk_003_labeled.csv: 395 missing (0 deleted/empty, 395 REAL gaps)
  ⚠️  AITA_chunk_004_labeled.csv: 361 missing (0 deleted/empty, 361 REAL gaps)
  ⚠️  AITA_chunk_005_labeled.csv: 389 missing (0 deleted/empty, 389 REAL gaps)
  ⚠️  AITA_chunk_006_labeled.csv: 473 missing (0 deleted/empty, 473 REAL gaps)
  ⚠️  AITA_chunk_007_labeled.csv: 317 missing (0 deleted/empty, 317 REAL gaps)
  ⚠️  AITA_chunk_008_labeled.csv: 282 missing (0 deleted/empty, 282 REAL gaps)
  ⚠️  AITA_chunk_009_labeled.csv: 311 missing (0 deleted/empty, 311 REAL gaps)
  ⚠️  CMV_chunk_000_labeled.cs

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RETRY #2: The final 40K stubborn rows
# ══════════════════════════════════════════════════════════════════════

import json, os, gc
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
LABELED_CHUNKS_DIR = os.path.join(BASE_DIR, "labeled_chunks")

retry2_jobs = []

for sub_dir in sorted(os.listdir(LABELED_CHUNKS_DIR)):
    sub_path = os.path.join(LABELED_CHUNKS_DIR, sub_dir)
    if not os.path.isdir(sub_path):
        continue

    sub_key = sub_dir.replace("_labeled", "")

    for labeled_file in sorted(os.listdir(sub_path)):
        if not labeled_file.endswith("_labeled.csv"):
            continue

        labeled_path = os.path.join(sub_path, labeled_file)
        df = pd.read_csv(labeled_path, low_memory=False)

        missing = df[df["confidence"] == 0.0].copy()

        if len(missing) < 10:
            continue

        chunk_name = labeled_file.replace("_labeled.csv", "")
        retry_name = f"{chunk_name}_retry2"

        print(f"  🔄 {chunk_name}: {len(missing):,} rows")

        jsonl_content = prepare_batch_jsonl(missing.reset_index(drop=True))

        if not jsonl_content.strip():
            print(f"     Empty JSONL — skipping")
            continue

        input_blob = f"batch_input_v2/{retry_name}.jsonl"
        input_uri = upload_to_gcs(BUCKET_NAME, input_blob, jsonl_content)
        output_uri = f"gs://{BUCKET_NAME}/{GCS_OUTPUT_PREFIX}/{retry_name}/"

        try:
            job = submit_batch_job(input_uri, output_uri)
            retry2_jobs.append({
                "sub": sub_key,
                "chunk": retry_name,
                "original_chunk": chunk_name,
                "job_name": job.name,
                "state": str(job.state),
                "missing_indices": missing.index.tolist(),
            })
            # Save after each submission (crash-safe)
            retry2_path = os.path.join(BASE_DIR, "batch_jobs_retry2.json")
            with open(retry2_path, "w") as f:
                json.dump(retry2_jobs, f, indent=2)
        except Exception as e:
            print(f"     ❌ Failed: {e}")

        del df, missing
        gc.collect()

print(f"\n{'═' * 60}")
print(f"  RETRY2 SUBMITTED: {len(retry2_jobs)} jobs, ~40K rows total")
print(f"  These will finish in minutes (tiny jobs).")
print(f"{'═' * 60}")

  🔄 AITA_chunk_000: 403 rows
  Uploaded: gs://project-41aa31b7-2463-46fd-963-dissent-batch/batch_input_v2/AITA_chunk_000_retry2.jsonl (1.0 MB, 403 requests)
  ✅ Job submitted: projects/92175944845/locations/global/batchPredictionJobs/3926624853381087232
     State: JobState.JOB_STATE_PENDING
  🔄 AITA_chunk_001: 531 rows
  Uploaded: gs://project-41aa31b7-2463-46fd-963-dissent-batch/batch_input_v2/AITA_chunk_001_retry2.jsonl (1.2 MB, 531 requests)
  ✅ Job submitted: projects/92175944845/locations/global/batchPredictionJobs/4251728451481894912
     State: JobState.JOB_STATE_PENDING
  🔄 AITA_chunk_002: 639 rows
  Uploaded: gs://project-41aa31b7-2463-46fd-963-dissent-batch/batch_input_v2/AITA_chunk_002_retry2.jsonl (1.5 MB, 639 requests)
  ✅ Job submitted: projects/92175944845/locations/global/batchPredictionJobs/359773948503654400
     State: JobState.JOB_STATE_PENDING
  🔄 AITA_chunk_003: 395 rows
  Uploaded: gs://project-41aa31b7-2463-46fd-963-dissent-batch/batch_input_v2/AITA_chunk_003_r

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CHECK STATUS — Retry2 jobs
# ══════════════════════════════════════════════════════════════════════

import json, os
from collections import Counter

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
retry2_path = os.path.join(BASE_DIR, "batch_jobs_retry2.json")

with open(retry2_path) as f:
    retry2_jobs = json.load(f)

print("═" * 65)
print(f"  RETRY2 JOBS ({len(retry2_jobs)})")
print("═" * 65)

counts = Counter()
rows_done = 0
rows_total = 0

for rj in retry2_jobs:
    job = client.batches.get(name=rj["job_name"])
    state = str(job.state).split(".")[-1]
    counts[state] += 1

    stats = job.completion_stats
    s = stats.successful_count or 0 if stats else 0
    i = stats.incomplete_count or 0 if stats else 0
    f_c = stats.failed_count or 0 if stats else 0
    total = s + i + f_c
    rows_done += s
    rows_total += total

    if state != "JOB_STATE_SUCCEEDED":
        pct = s / total * 100 if total > 0 else 0
        print(f"  {rj['chunk']:35s} {state:25s} {s:>6,}/{total:>6,} ({pct:5.1f}%)")

print(f"\n  ✅ Succeeded: {counts.get('JOB_STATE_SUCCEEDED', 0)}/{len(retry2_jobs)}")
print(f"  ⏳ Pending:   {counts.get('JOB_STATE_PENDING', 0)}/{len(retry2_jobs)}")
print(f"  🔄 Running:   {counts.get('JOB_STATE_RUNNING', 0)}/{len(retry2_jobs)}")
print(f"  ❌ Failed:    {counts.get('JOB_STATE_FAILED', 0)}/{len(retry2_jobs)}")
print(f"\n  Rows: {rows_done:,}/{rows_total:,} completed")

if counts.get("JOB_STATE_SUCCEEDED", 0) == len(retry2_jobs):
    print(f"\n  🎉 ALL DONE! Run the merge cell next.")

═════════════════════════════════════════════════════════════════
  RETRY2 JOBS (50)
═════════════════════════════════════════════════════════════════

  ✅ Succeeded: 50/50
  ⏳ Pending:   0/50
  🔄 Running:   0/50
  ❌ Failed:    0/50

  Rows: 41,108/41,108 completed

  🎉 ALL DONE! Run the merge cell next.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MERGE RETRY2 RESULTS back into the labeled CSVs
# ══════════════════════════════════════════════════════════════════════

import json, os, gc
import pandas as pd
from google.cloud import storage

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
LABELED_CHUNKS_DIR = os.path.join(BASE_DIR, "labeled_chunks")

retry2_path = os.path.join(BASE_DIR, "batch_jobs_retry2.json")
with open(retry2_path) as f:
    retry2_jobs = json.load(f)

LABEL_MAP = {
    "substantive_dissent": 0,
    "agreement": 1,
    "neutral": 2,
    "social_disagreement": 3,
}

storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)

patched = 0
still_missing_total = 0

for rj in retry2_jobs:
    sub = rj["sub"]
    retry_name = rj["chunk"]
    original_chunk = rj["original_chunk"]
    job_name = rj["job_name"]
    missing_indices = rj["missing_indices"]

    job = client.batches.get(name=job_name)
    state = str(job.state).split(".")[-1]
    if state not in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED"):
        print(f"  ⏭️  {retry_name}: {state} — not done yet")
        continue

    labeled_path = os.path.join(
        LABELED_CHUNKS_DIR, f"{sub}_labeled", f"{original_chunk}_labeled.csv"
    )
    if not os.path.exists(labeled_path):
        print(f"  ❌ {original_chunk}: labeled CSV not found")
        continue

    df = pd.read_csv(labeled_path, low_memory=False)

    output_prefix = f"{GCS_OUTPUT_PREFIX}/{retry_name}/"
    blobs = list(bucket.list_blobs(prefix=output_prefix))
    jsonl_blobs = [b for b in blobs if b.name.endswith(".jsonl")]

    if not jsonl_blobs:
        print(f"  ❌ {retry_name}: no output JSONL found")
        continue

    retry_labels = {}
    parse_errors = 0

    for blob in jsonl_blobs:
        content = blob.download_as_text()
        for line in content.strip().split("\n"):
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
                retry_row_num = int(obj.get("metadata", -1))
                if retry_row_num < 0 or retry_row_num >= len(missing_indices):
                    parse_errors += 1
                    continue

                response = obj.get("response", {})
                candidates = response.get("candidates", [])
                if not candidates:
                    parse_errors += 1
                    continue

                parts = candidates[0].get("content", {}).get("parts", [])
                if not parts:
                    parse_errors += 1
                    continue

                text = parts[0].get("text", "").strip()
                parsed = json.loads(text)
                raw_label = parsed.get("label", "neutral").lower().strip()
                confidence = float(parsed.get("confidence", 0.5))

                if raw_label not in LABEL_MAP:
                    matched = False
                    for key in LABEL_MAP:
                        if key in raw_label or raw_label in key:
                            raw_label = key
                            matched = True
                            break
                    if not matched:
                        raw_label = "neutral"
                        confidence = 0.0

                original_idx = missing_indices[retry_row_num]
                retry_labels[original_idx] = {
                    "label": LABEL_MAP[raw_label],
                    "label_name": raw_label,
                    "confidence": confidence,
                }
            except (json.JSONDecodeError, KeyError, ValueError, IndexError):
                parse_errors += 1

    filled = 0
    for orig_idx, label_info in retry_labels.items():
        if orig_idx < len(df):
            df.at[orig_idx, "label"] = label_info["label"]
            df.at[orig_idx, "label_name"] = label_info["label_name"]
            df.at[orig_idx, "confidence"] = label_info["confidence"]
            filled += 1

    still_missing = (df["confidence"] == 0.0).sum()
    still_missing_total += still_missing

    df.to_csv(labeled_path, index=False)
    patched += 1

    print(f"  ✅ {original_chunk}: patched {filled:,} rows, "
          f"{still_missing:,} still missing, {parse_errors} parse errors")

    del df
    gc.collect()

print(f"\n{'═' * 60}")
print(f"  MERGE COMPLETE: {patched} chunks patched")
print(f"  Still missing across all chunks: {still_missing_total:,}")
print(f"{'═' * 60}")

  ✅ AITA_chunk_000: patched 403 rows, 314 still missing, 0 parse errors
  ✅ AITA_chunk_001: patched 531 rows, 435 still missing, 0 parse errors
  ✅ AITA_chunk_002: patched 639 rows, 521 still missing, 0 parse errors
  ✅ AITA_chunk_003: patched 395 rows, 317 still missing, 0 parse errors
  ✅ AITA_chunk_004: patched 361 rows, 298 still missing, 0 parse errors
  ✅ AITA_chunk_005: patched 389 rows, 323 still missing, 0 parse errors
  ✅ AITA_chunk_006: patched 473 rows, 390 still missing, 0 parse errors
  ✅ AITA_chunk_007: patched 317 rows, 243 still missing, 0 parse errors
  ✅ AITA_chunk_008: patched 282 rows, 220 still missing, 0 parse errors
  ✅ AITA_chunk_009: patched 311 rows, 265 still missing, 0 parse errors
  ✅ AITA_chunk_010: patched 536 rows, 408 still missing, 0 parse errors
  ✅ AITA_chunk_011: patched 566 rows, 403 still missing, 0 parse errors
  ✅ CMV_chunk_000: patched 361 rows, 308 still missing, 0 parse errors
  ✅ CMV_chunk_001: patched 517 rows, 434 still missing, 0 parse e

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FINAL COVERAGE REPORT — the numbers you'd put in your paper
# ══════════════════════════════════════════════════════════════════════

import os, gc
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
LABELED_CHUNKS_DIR = os.path.join(BASE_DIR, "labeled_chunks")

sub_stats = {}
total_rows = 0
total_labeled = 0
total_missing = 0

for sub_dir in sorted(os.listdir(LABELED_CHUNKS_DIR)):
    sub_path = os.path.join(LABELED_CHUNKS_DIR, sub_dir)
    if not os.path.isdir(sub_path):
        continue

    sub_key = sub_dir.replace("_labeled", "")
    sub_rows = 0
    sub_labeled = 0
    sub_missing = 0

    for f in sorted(os.listdir(sub_path)):
        if not f.endswith("_labeled.csv"):
            continue
        df = pd.read_csv(os.path.join(sub_path, f), low_memory=False)
        n = len(df)
        n_missing = (df["confidence"] == 0.0).sum()
        n_labeled = n - n_missing
        sub_rows += n
        sub_labeled += n_labeled
        sub_missing += n_missing
        del df; gc.collect()

    sub_stats[sub_key] = {
        "rows": sub_rows,
        "labeled": sub_labeled,
        "missing": sub_missing,
    }
    total_rows += sub_rows
    total_labeled += sub_labeled
    total_missing += sub_missing

print("═" * 65)
print("  FINAL LABELING COVERAGE REPORT")
print("═" * 65)
print(f"  {'Subreddit':<15} {'Total':>10} {'Labeled':>10} {'Missing':>8} {'Coverage':>9}")
print(f"  {'─'*15} {'─'*10} {'─'*10} {'─'*8} {'─'*9}")

for sub, s in sub_stats.items():
    pct = s["labeled"] / s["rows"] * 100 if s["rows"] > 0 else 0
    print(f"  {sub:<15} {s['rows']:>10,} {s['labeled']:>10,} {s['missing']:>8,} {pct:>8.2f}%")

pct_total = total_labeled / total_rows * 100
print(f"  {'─'*15} {'─'*10} {'─'*10} {'─'*8} {'─'*9}")
print(f"  {'TOTAL':<15} {total_rows:>10,} {total_labeled:>10,} {total_missing:>8,} {pct_total:>8.2f}%")

print(f"\n  📝 For your methodology section:")
print(f"     'Of {total_rows:,} comment-reply pairs, {total_labeled:,} ({pct_total:.1f}%)")
print(f"      were successfully classified. The remaining {total_missing:,} rows")
print(f"      ({100-pct_total:.1f}%) received no valid model response after multiple")
print(f"      retry attempts (likely due to content safety filtering) and")
print(f"      were excluded from analysis.'")
print(f"\n  ⚠️  Do NOT default them to neutral. EXCLUDE them:")
print(f"      df = df[df['confidence'] > 0]")

═════════════════════════════════════════════════════════════════
  FINAL LABELING COVERAGE REPORT
═════════════════════════════════════════════════════════════════
  Subreddit            Total    Labeled  Missing  Coverage
  ─────────────── ────────── ────────── ──────── ─────────
  AITA               586,657    582,520    4,137    99.29%
  CMV                577,825    574,753    3,072    99.47%
  POLOP               86,493     86,094      399    99.54%
  T10D               594,278    578,846   15,432    97.40%
  UNPOPULAR          580,579    569,610   10,969    98.11%
  ─────────────── ────────── ────────── ──────── ─────────
  TOTAL            2,425,832  2,391,823   34,009    98.60%

  📝 For your methodology section:
     'Of 2,425,832 comment-reply pairs, 2,391,823 (98.6%)
      were successfully classified. The remaining 34,009 rows
      (1.4%) received no valid model response after multiple
      retry attempts (likely due to content safety filtering) and
      were excluded fr

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Verification: Check label distribution from completed chunks
# ══════════════════════════════════════════════════════════════════════

BASE_DIR = "/content/drive/MyDrive/My_Dissent_project"
LABELED_CHUNKS_DIR = os.path.join(BASE_DIR, "labeled_chunks")

if not os.path.exists(LABELED_CHUNKS_DIR):
    print("No labeled data yet. Run the harvest cell first.")
else:
    for sub_dir in sorted(os.listdir(LABELED_CHUNKS_DIR)):
        sub_path = os.path.join(LABELED_CHUNKS_DIR, sub_dir)
        if not os.path.isdir(sub_path):
            continue

        labeled_files = sorted([f for f in os.listdir(sub_path) if f.endswith("_labeled.csv")])
        if not labeled_files:
            continue

        first_file = os.path.join(sub_path, labeled_files[0])
        df = pd.read_csv(first_file, low_memory=False)

        print(f"\n{'─' * 60}")
        print(f"  {sub_dir} — {labeled_files[0]} ({len(df):,} rows)")
        print(f"{'─' * 60}")
        print(f"  Label distribution:")
        print(df['label_name'].value_counts().to_string())
        print(f"\n  Mean confidence: {df['confidence'].mean():.3f}")
        print(f"  Low confidence (<0.5): {(df['confidence'] < 0.5).sum():,} "
              f"({(df['confidence'] < 0.5).mean()*100:.1f}%)")

        for label in sorted(df['label_name'].unique()):
            example = df[df['label_name'] == label].iloc[0]
            print(f"\n  Example [{label}] (conf={example['confidence']:.2f}):")
            print(f"    Parent: {str(example['parent_body'])[:100]}...")
            print(f"    Reply:  {str(example['comment_body'])[:100]}...")


────────────────────────────────────────────────────────────
  AITA_labeled — AITA_chunk_000_labeled.csv (48,330 rows)
────────────────────────────────────────────────────────────
  Label distribution:
label_name
substantive_dissent    29462
agreement              11458
neutral                 4438
social_disagreement     2972

  Mean confidence: 0.905
  Low confidence (<0.5): 314 (0.6%)

  Example [agreement] (conf=0.90):
    Parent: Me (34F) and my ex-boss (60F) have become pretty close since I started at my job in 2013. It's a gov...
    Reply:  Wow. This relationship is… strange? You are definitely NTA but I would really distance myself from t...

  Example [neutral] (conf=0.98):
    Parent: Me (34F) and my ex-boss (60F) have become pretty close since I started at my job in 2013. It's a gov...
    Reply:  Welcome to /r/AmITheAsshole. Please view our [voting guide here](https://www.reddit.com/r/AmItheAssh...

  Example [social_disagreement] (conf=0.91):
    Parent: Why would they b